# Modern Tokenizers: Special Tokens and Chat Templates

This notebook explores how modern language models use special tokens and format conversations.

We'll explore:
- GPT-4 (OpenAI) - using tiktoken
- Qwen (Alibaba) - using transformers
- Claude special tokens (conceptual)

In [ ]:
# Install if needed
# !pip install tiktoken transformers

In [ ]:
import tiktoken
from transformers import AutoTokenizer

## Part 1: Special Tokens in Modern Models

Different models use different special tokens to structure conversations.

### GPT-4 Special Tokens

In [ ]:
# Load GPT-4 tokenizer
gpt4_enc = tiktoken.get_encoding("cl100k_base")

print("GPT-4 Special Tokens:\n")
gpt4_special = {
    '<|endoftext|>': 100257,
    '<|fim_prefix|>': 100258,
    '<|fim_middle|>': 100259,
    '<|fim_suffix|>': 100260,
    '<|endofprompt|>': 100276
}

for token, id in gpt4_special.items():
    print(f"  {token:<20} → ID {id}")

print(f"\nVocabulary size: {gpt4_enc.n_vocab:,}")

### Qwen Special Tokens

In [ ]:
# Load Qwen tokenizer
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

print("Qwen Special Tokens:\n")
print(f"  BOS (Beginning): {qwen_tokenizer.bos_token} (ID: {qwen_tokenizer.bos_token_id})")
print(f"  EOS (End):       {qwen_tokenizer.eos_token} (ID: {qwen_tokenizer.eos_token_id})")
print(f"  PAD (Padding):   {qwen_tokenizer.pad_token} (ID: {qwen_tokenizer.pad_token_id})")

print(f"\n  Additional special tokens:")
for token in qwen_tokenizer.additional_special_tokens[:10]:  # Show first 10
    token_id = qwen_tokenizer.convert_tokens_to_ids(token)
    print(f"    {token:<20} → ID {token_id}")

print(f"\nVocabulary size: {len(qwen_tokenizer):,}")

### Claude Special Tokens (Conceptual)

Claude uses similar special tokens:
- `<|im_start|>` - Start of a message
- `<|im_end|>` - End of a message  
- `<think>` / `</think>` - Internal reasoning (for some models)

**Note**: Claude's tokenizer is not publicly available, but the API handles this automatically.

---

## Part 2: How Special Tokens Behave

### Exercise 1: Encoding vs Decoding

**Task**: What happens when you try to encode special tokens as regular text?

In [ ]:
# Try encoding special token as regular text
text_with_special = "Hello <|endoftext|> world"

print("Text:", text_with_special)
print("\nGPT-4 tokenization:")
tokens = gpt4_enc.encode(text_with_special)
print(f"  Token IDs: {tokens}")
print(f"  Decoded: {gpt4_enc.decode(tokens)}")

---

### Solution 1: Special Tokens Need Special Handling

In [ ]:
# Special tokens are NOT treated specially by default
# They're tokenized like regular text: "<", "|", "end", "of", "text", "|", ">"

print("Breaking down the tokenization:")
for token_id in tokens:
    token_bytes = gpt4_enc.decode_single_token_bytes(token_id)
    token_str = token_bytes.decode('utf-8', errors='replace')
    print(f"  ID {token_id:5d} → '{token_str}'")

print("\nTo use special tokens, you need to insert their IDs directly!")

---

## Part 3: Chat Templates

Modern chat models format conversations with special tokens to distinguish:
- System instructions
- User messages
- Assistant responses

### Qwen Chat Format

In [ ]:
# Example conversation
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user", "content": "What is its population?"}
]

### Exercise 2: Applying Chat Template

**Task**: How do you think the model formats this conversation with special tokens?

*(Think for 30 seconds)*

In [ ]:
# Apply chat template
formatted = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("Formatted conversation:")
print(formatted)

---

### Solution 2: Understanding the Format

In [ ]:
# Let's tokenize and see the token IDs
token_ids = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)

print(f"Total tokens: {len(token_ids)}\n")
print("First 30 tokens:")
for i, token_id in enumerate(token_ids[:30]):
    token = qwen_tokenizer.decode([token_id])
    print(f"  {i:3d}: ID {token_id:6d} → '{token}'")

### With Generation Prompt

In [ ]:
# Add generation prompt (prepares for model to respond)
formatted_with_prompt = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("With generation prompt:")
print(formatted_with_prompt)
print("\nNotice the format is ready for the assistant to continue!")

---

## Part 4: GPT-4 Chat Format (Conceptual)

GPT-4 uses a similar approach through the OpenAI API, but `tiktoken` doesn't have `apply_chat_template`.

The API handles this internally:
```python
# OpenAI API format
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"}
    ]
)
```

Internally, it formats with special tokens similar to what we saw with Qwen.

---

## Part 5: Comparing Token Efficiency

Different models have different chat formats, which affects token usage.

In [ ]:
# Count tokens for the same conversation
print("Token counts for same conversation:\n")

# Qwen
qwen_tokens = qwen_tokenizer.apply_chat_template(messages, tokenize=True)
print(f"Qwen:  {len(qwen_tokens)} tokens")

# GPT-4 (approximate - just the raw text)
raw_text = "\n".join([f"{m['role']}: {m['content']}" for m in messages])
gpt4_tokens = gpt4_enc.encode(raw_text)
print(f"GPT-4 (approx): {len(gpt4_tokens)} tokens")
print("\nNote: Actual GPT-4 API adds more tokens for proper chat formatting")

---

## Part 6: Decoding Behavior with Special Tokens

### Exercise 3: What Happens When Decoding?

**Task**: What do you think happens when you decode token IDs that include special tokens?

In [ ]:
# Decode with and without special tokens
token_ids = qwen_tokenizer.apply_chat_template(messages, tokenize=True)

print("Decoded WITH special tokens:")
decoded_with = qwen_tokenizer.decode(token_ids, skip_special_tokens=False)
print(decoded_with)
print("\n" + "="*60 + "\n")

print("Decoded WITHOUT special tokens:")
decoded_without = qwen_tokenizer.decode(token_ids, skip_special_tokens=True)
print(decoded_without)

---

### Solution 3: Skip Special Tokens for Clean Text

In [ ]:
print("Key insights:\n")
print("1. With special tokens: You see the internal structure")
print("   - Useful for debugging")
print("   - Shows exactly what the model sees\n")

print("2. Without special tokens: Clean, readable text")
print("   - Useful for display to users")
print("   - Hides implementation details\n")

print("Always use skip_special_tokens=True when showing text to users!")

---

## Part 7: Custom Chat Template

You can inspect and even customize the chat template.

In [ ]:
# View the chat template
print("Qwen's chat template:\n")
if hasattr(qwen_tokenizer, 'chat_template'):
    print(qwen_tokenizer.chat_template[:500])  # First 500 chars
    print("\n... (truncated)")
else:
    print("No chat template found")

---

## Part 8: Practical Example - Building a Chat

Let's simulate a multi-turn conversation.

In [ ]:
# Start with system message
conversation = [
    {"role": "system", "content": "You are a Python programming tutor."}
]

def add_turn(user_msg, assistant_msg=None):
    """Add a conversation turn and show token count"""
    conversation.append({"role": "user", "content": user_msg})
    if assistant_msg:
        conversation.append({"role": "assistant", "content": assistant_msg})
    
    tokens = qwen_tokenizer.apply_chat_template(conversation, tokenize=True)
    print(f"Turn {len(conversation)//2}: {len(tokens)} tokens total")
    return tokens

# Simulate conversation
print("Building conversation:\n")
add_turn("What is a list in Python?")
add_turn(
    "What is a list in Python?",
    "A list is a built-in data structure that stores an ordered collection of items."
)
add_turn(
    "How do I add items?",
    "You can use the append() method: my_list.append(item)"
)

print("\nToken usage grows with conversation length!")

---

## Summary: Key Takeaways

1. **Special tokens structure conversations**
   - Different models use different special tokens
   - GPT-4: `<|endoftext|>`, `<|fim_*|>`, etc.
   - Qwen: `<|im_start|>`, `<|im_end|>`, etc.

2. **Chat templates format multi-turn conversations**
   - `apply_chat_template()` handles this automatically
   - Distinguishes system/user/assistant roles
   - Adds special tokens in the right places

3. **Decoding behavior**
   - `skip_special_tokens=False`: See internal structure
   - `skip_special_tokens=True`: Clean text for users

4. **Token efficiency matters**
   - Chat formatting adds overhead tokens
   - Longer conversations = more tokens
   - Different models have different overhead

5. **Practical implications**:
   - Always check token counts for cost estimation
   - Use appropriate decoding for your use case
   - Understand your model's chat format